# Stage 11 V2d — Canonical CPU Restore + GPU Notehead Semantic Rerun

Restore remains on the exact canonical CPU profile. Only the measurement-only Oemer `seg_net` notehead detector uses GPU. Before the full sweep, the GPU detector must reproduce the durable CPU canary notehead boxes **exactly** for both source and restored Beethoven p2; otherwise the run stops. No training, held-out access, production promotion, or Stage 12.

**Use a GPU runtime.** The first cell now performs a zero-cost GPU attachment preflight before installing anything.


In [ ]:
# 1) Zero-cost GPU attachment preflight, then build exact CPU-Restore + GPU-detector runtime.
import platform, shutil, subprocess, sys
from pathlib import Path
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is None:
    raise RuntimeError('GPU runtime is not attached. In Colab choose Runtime > Change runtime type > Hardware accelerator > T4 GPU, save, then run all from the top.')
probe = subprocess.run([nvidia_smi, '--query-gpu=name,driver_version', '--format=csv,noheader'], text=True, capture_output=True)
if probe.returncode != 0 or not probe.stdout.strip():
    raise RuntimeError('GPU runtime probe failed before setup. Reconnect with a T4 GPU runtime and run all again. Details: ' + (probe.stderr.strip() or 'no GPU reported'))
print('GPU RUNTIME PREFLIGHT PASS:', probe.stdout.strip())
print('colab system python', platform.python_version())
subprocess.run([sys.executable,'-m','pip','install','-q','uv==0.12.10'], check=True)
VENV=Path('/content/st-score-restore-canonical-gpu-detector-py3135')
subprocess.run(['uv','venv','--python','3.13.5','--clear',str(VENV)], check=True)
PY=VENV/'bin'/'python'
subprocess.run(['uv','pip','install','--python',str(PY),'numpy>=2.0,<2.3','opencv-python-headless','filelock','typing-extensions','sympy','networkx','jinja2','fsspec','onnxruntime-gpu==1.20.2'], check=True)
subprocess.run(['uv','pip','install','--python',str(PY),'--index-url','https://download.pytorch.org/whl/cpu','torch==2.10.0'], check=True)
verify=r'''import platform, torch, onnxruntime as ort, numpy as np
print('python',platform.python_version())
print('torch',torch.__version__,'torch_cuda',torch.cuda.is_available())
print('ort',ort.__version__,ort.get_available_providers())
assert platform.python_version()=='3.13.5'
assert torch.__version__=='2.10.0+cpu'
assert not torch.cuda.is_available()
assert ort.__version__=='1.20.2'
assert 'CUDAExecutionProvider' in ort.get_available_providers()
assert tuple(map(int,np.__version__.split('.')[:2])) < (2,3)
'''
subprocess.run([str(PY),'-c',verify], check=True)
print('EXACT CPU RESTORE + GPU DETECTOR RUNTIME READY')


In [ ]:
# 2) Mount durable Drive cache, pin reviewed repository code, and prove CUDA session creation.
from google.colab import drive
drive.mount('/content/drive')
import shutil, subprocess
from pathlib import Path
CACHE=Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_CACHE_V1')
RESULTS=Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_RESULTS')
CPU_RUN=CACHE/'canonical_cpu_notehead_semantic_v1'
required=[CACHE/'cache_manifest.json',CACHE/'exact_inputs'/'v2a_candidate_512.torchscript.pt',CACHE/'oemer_checkpoints'/'seg_net'/'model.onnx',RESULTS/'v2d_colab_gpu_detector_benchmark_result.json',CPU_RUN/'manifest.json',CPU_RUN/'restored_pages'/'beethoven-op48-no3-p2.png',CPU_RUN/'source_notehead_detector'/'beethoven-op48-no3-p2.json',CPU_RUN/'restored_notehead_detector'/'beethoven-op48-no3-p2.json']
for path in required:
    if not path.exists(): raise FileNotFoundError(path)
REPO=Path('/content/st-score-restore-engine')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git',str(REPO)], check=True)
EXPECTED_COMMIT='30bce4cb4e4c0b321288f176db13b736ba61de5e'
subprocess.run(['git','-C',str(REPO),'checkout','-q',EXPECTED_COMMIT], check=True)
actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert actual==EXPECTED_COMMIT
checkpoint=CACHE/'oemer_checkpoints'/'seg_net'/'model.onnx'
session_check=r'''import onnxruntime as ort,sys
p=sys.argv[1]
s=ort.InferenceSession(p,providers=['CUDAExecutionProvider','CPUExecutionProvider'])
print('session providers',s.get_providers())
assert s.get_providers()[0]=='CUDAExecutionProvider'
assert len(s.get_inputs())==1 and len(s.get_outputs())>=1
'''
subprocess.run([str(PY),'-c',session_check,str(checkpoint)], check=True)
print('DRIVE + PINNED REPO + CUDA SESSION READY',actual)


In [ ]:
# 3) Run exact CPU-canary↔GPU-detector gate, then full rerun only if equivalent and within budget.
import os, subprocess
LOG=RESULTS/'v2d_canonical_cpu_restore_gpu_notehead_semantic_rerun.log'
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['PYTHONFAULTHANDLER']='1'; env['PYTHONPATH']=str(REPO/'src')
cmd=[str(PY),'-m','st_score_restore.stage11_v2d_canonical_cpu_restore_gpu_notehead_semantic_rerun','--run']
print('Restore: CPU only | Detector: CUDA GPU')
print('running:',' '.join(cmd))
print('log:',LOG)
with LOG.open('a',encoding='utf-8',buffering=1) as log:
    log.write('\n===== CANONICAL CPU RESTORE + GPU NOTEHEAD SEMANTIC RERUN =====\n')
    proc=subprocess.Popen(cmd,cwd=REPO,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line,end=''); log.write(line)
    rc=proc.wait()
if rc!=0: raise subprocess.CalledProcessError(rc,cmd)


## Expected markers

`GPU RUNTIME PREFLIGHT PASS` → `EXACT CPU RESTORE + GPU DETECTOR RUNTIME READY` → `DRIVE + PINNED REPO + CUDA SESSION READY` → `CANONICAL CPU RESTORE + GPU DETECTOR PREFLIGHT PASS` → `CPU↔GPU CANARY EQUIVALENCE PASS` → `GPU COST BUDGET PASS` → `CANONICAL CPU RESTORE + GPU NOTEHEAD SEMANTIC RERUN COMPLETE` → `SAVED:`

If the first cell says GPU runtime is not attached, select a T4 GPU runtime before continuing. If CPU↔GPU equivalence fails, or the projected 18-page GPU detector time exceeds 45 minutes, the run stops before the full sweep and preserves completed cache.
